- **Kelas** : IF403
- **Program Studi** : PJJ Informatika
- **Nama Mahasiswa** : Raka Anggie Saputra
- **NIM** : 240401010148


In [5]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [2]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [6]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


In [7]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_items,
    metric='confidence',
    min_threshold=0.5
)

rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(
    rules[['antecedents', 'consequents',
           'support', 'confidence', 'lift']].head(10)
)

         antecedents consequents  support  confidence      lift
9        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
14  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
15     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
13   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


## Interpretasi Hasil Association Rules

Berdasarkan hasil Association Rules yang diperoleh, aturan dengan **nilai Lift tertinggi** adalah:

- **{Keju, Teh} → {Telur}**
  - Support = **0.12**
  - Confidence = **0.857**
  - Lift = **2.381**

Artinya, pada 12% dari seluruh transaksi, ketiga produk tersebut muncul bersama. Selain itu, jika pelanggan membeli **Keju dan Teh**, maka terdapat peluang sekitar **85,7%** bahwa pelanggan juga membeli **Telur**. Nilai **Lift > 1** menunjukkan bahwa hubungan tersebut lebih kuat dibandingkan jika pembelian terjadi secara acak.

Aturan yang juga menarik adalah:

- **{Gula, Roti} → {Selai}**
  - Confidence = **1.000**
  - Lift = **1.923**

Artinya, setiap transaksi yang mengandung **Gula** dan **Roti** pada dataset ini selalu disertai dengan **Selai**.

Selain itu, aturan:

- **{Roti} → {Selai}**
  - Support = **0.22**
  - Confidence = **0.688**
  - Lift = **1.322**

menunjukkan bahwa sekitar **68,8%** transaksi yang membeli **Roti** juga membeli **Selai**. Hasil ini masuk akal karena pada proses pembuatan dataset memang disisipkan pola bahwa **Roti sering dibeli bersama Selai**.

### Kesimpulan

- Aturan dengan **Lift tertinggi** adalah **{Keju, Teh} → {Telur}**.
- Hubungan **Roti → Selai** juga terlihat cukup kuat dan sesuai dengan pola yang sengaja dibuat pada dataset.
- Nilai **Lift** yang lebih besar dari 1 pada seluruh aturan menunjukkan adanya hubungan positif antarproduk, sehingga aturan-aturan tersebut dapat dimanfaatkan sebagai rekomendasi produk atau strategi promosi (cross-selling).

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i[0] for i in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [9]:
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x
)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


## Perbandingan Association Rules dan Content-Based Filtering

### Hasil Rekomendasi

**Association Rules**
- Selai (Lift = 1.923)
- Selai (Lift = 1.322)

**Content-Based Filtering**
- Selai
- Sereal
- Susu

### Apakah kedua pendekatan memberi rekomendasi yang konsisten?

Ya, kedua pendekatan memberikan rekomendasi yang **konsisten pada produk Selai**. Hal ini menunjukkan bahwa baik berdasarkan pola pembelian pelanggan maupun berdasarkan kemiripan kategori produk, **Selai merupakan produk yang paling relevan untuk direkomendasikan kepada pelanggan yang membeli Roti**.

Namun, terdapat perbedaan pada rekomendasi berikutnya. Association Rules hanya merekomendasikan **Selai** karena rekomendasi didasarkan pada pola transaksi yang benar-benar sering muncul bersama. Sebaliknya, Content-Based Filtering juga merekomendasikan **Sereal** dan **Susu** karena metode ini melihat kemiripan karakteristik atau kategori produk, bukan riwayat pembelian pelanggan.

### Kapan sebaiknya menggunakan masing-masing pendekatan?

- **Association Rules** lebih cocok digunakan ketika tersedia data transaksi yang cukup banyak dan tujuan utamanya adalah menemukan produk yang sering dibeli secara bersamaan (cross-selling atau market basket analysis).

- **Content-Based Filtering** lebih cocok digunakan ketika data transaksi masih terbatas atau ketika ingin merekomendasikan produk yang memiliki karakteristik serupa dengan produk yang sedang dilihat atau dibeli oleh pelanggan.

### Kesimpulan

Pada Pertemuan 12 dibahas **sistem rekomendasi produk** menggunakan dua pendekatan, yaitu **Association Rules** dan **Content-Based Filtering**.

Pada Association Rules, digunakan 50 transaksi dengan beberapa produk seperti Roti, Selai, Susu, Sereal, Telur, Keju, Kopi, Gula, Teh, dan Mentega. Hasil analisis menunjukkan aturan dengan **Lift tertinggi** adalah:

> **{Keju, Teh} → {Telur}**

dengan:

* Support = **0,12**
* Confidence = **0,857**
* Lift = **2,381**

Selain itu, pola **Roti → Selai** juga cukup kuat dengan confidence sekitar **68,8%** dan lift **1,322**. 

Kemudian digunakan Content-Based Filtering berdasarkan **kategori produk**. Untuk produk **Roti**, rekomendasi yang dihasilkan adalah:

* Selai
* Sereal
* Susu

Association Rules dan Content-Based Filtering sama-sama merekomendasikan **Selai** untuk pelanggan yang membeli Roti. 

**Kesimpulan utama:**
Association Rules menghasilkan rekomendasi berdasarkan **pola pembelian nyata pelanggan**, sedangkan Content-Based Filtering berdasarkan **kemiripan karakteristik atau kategori produk**. Kedua metode dapat menghasilkan rekomendasi yang saling melengkapi. Oleh karena itu, notebook menyimpulkan bahwa penerapan **hybrid recommendation system** yang menggabungkan kedua pendekatan dapat menghasilkan rekomendasi yang lebih baik. 
